# InsightAid AI — Sentiment Model Training (Objective 1)

Trains and evaluates all three sentiment models (XLM-R, mBERT, TF-IDF +
Logistic Regression) against the real `insightaid_sentiment` dataset
prepared by `01_sentiment_data_preparation.ipynb`
(`dataset_version` recorded in `manifest.json`), across both evaluation
paths (original-language, translate-to-English), using the already-tested
code in `ml/sentiment/src/` — no training logic is duplicated here.

**Before running:** Runtime → Change runtime type → **T4 GPU**. XLM-R/mBERT
will be extremely slow on CPU.

**Disconnect-resilient by design**: every cell below calls `bootstrap()`
first and checks Drive for an already-trained checkpoint before training
anything. If Colab disconnects you partway through (common on a run this
long), you don't need to figure out which cells to re-run — just
**Runtime → Run all** again. Finished steps are skipped (cheap checks),
unfinished ones resume from scratch. Nothing here invents a metric: every
number comes from `ml/sentiment/src/metrics.py` run against this model's
real predictions on the real held-out test set.

In [1]:
import shutil
shutil.rmtree('/content/insightaid-ai', ignore_errors=True)



In [2]:
# Bootstrap — mounts Drive, gets the repo code, installs deps, loads the
# dataset. Idempotent: safe to call from the top of every cell below, since
# there's no way to know from inside one cell whether a disconnect wiped
# everything since the last one ran.

import os, sys

# Confirmed via a real Drive directory listing — NOT "insightaid_sentiment_data"
# (that was an incorrect assumption in an earlier version of this notebook).
DATA_DIR = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment"
REPO_DIR = "/content/insightaid-ai"
CKPT_DIR = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment_checkpoints"  # Drive-backed, NOT /content — survives disconnects


def bootstrap():
    global train_df, val_df, test_df, DATASET_VERSION, TEXT_COL, LABEL_COL

    try:
        import google.colab  # noqa: F401
        in_colab = True
    except ImportError:
        in_colab = False

    if in_colab and not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        # Force remount Google Drive to handle stuck connections
        drive.mount('/content/drive', force_remount=True)

    assert os.path.exists(DATA_DIR), (
        f"Dataset folder not found: {DATA_DIR}. If your Drive layout differs, "
        f"search for it first: os.walk('/content/drive/MyDrive') looking for 'manifest.json', "
        f"then update DATA_DIR above to match."
    )
    os.makedirs(CKPT_DIR, exist_ok=True)

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f'git clone --depth 1 https://github.com/danishalif04/InsightAID-AI.git {REPO_DIR}')
    sentiment_src = os.path.join(REPO_DIR, "ml", "sentiment")
    if sentiment_src not in sys.path:
        sys.path.insert(0, sentiment_src)

    try:
        import torch  # noqa: F401
        import transformers  # noqa: F401
    except ImportError:
        get_ipython().system('pip install -q torch transformers datasets accelerate sentencepiece sacremoses huggingface_hub')

    if 'train_df' not in globals():
        import json as _json
        import pandas as pd
        with open(os.path.join(DATA_DIR, "manifest.json")) as f:
            manifest = _json.load(f)
        DATASET_VERSION = manifest["dataset_version"]
        train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
        val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
        test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
        TEXT_COL, LABEL_COL = "text", "label"
        print(f"Loaded dataset_version={DATASET_VERSION}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


bootstrap()
print("Ready. DATA_DIR =", DATA_DIR, "| CKPT_DIR =", CKPT_DIR)

Mounted at /content/drive
Cloning into '/content/insightaid-ai'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (299/299), done.
remote: Total 323 (delta 12), reused 267 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (323/323), 451.45 KiB | 5.25 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Loaded dataset_version=v20261012-ba90cb55d8f7  train=85639  val=10763  test=10769
Ready. DATA_DIR = /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment | CKPT_DIR = /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment_checkpoints


In [3]:
# GPU check — fix this before training XLM-R/mBERT, don't just note it and continue.
bootstrap()
import torch
print("GPU available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True - Tesla T4


In [4]:
# Train the TF-IDF + Logistic Regression baseline. Fast (seconds), CPU-only
# — cheap enough that it just always retrains rather than checkpointing.
bootstrap()
from src.train_tfidf_baseline import train_tfidf_baseline, predict as tfidf_predict

tfidf_model = train_tfidf_baseline(train_df[TEXT_COL], train_df[LABEL_COL])
print("TF-IDF baseline trained on", len(train_df), "rows.")

TF-IDF baseline trained on 85639 rows.


In [5]:
# Train XLM-RoBERTa (primary model, FYP2.md §46). Needs the GPU runtime.
# Checks Drive for an already-trained checkpoint first — training this
# again from scratch after a disconnect would waste real GPU time for no
# reason if it already finished once.
bootstrap()
from pathlib import Path
import torch
from src.train_xlmr import XLMR_CHECKPOINT, TrainedModel, train_transformer, predict as xlmr_predict
from transformers import AutoModelForSequenceClassification, AutoTokenizer

XLMR_DRIVE_DIR = os.path.join(CKPT_DIR, "xlmr_model")

if 'xlmr_trained' in globals():
    print("xlmr_trained already in memory — skipping.")
elif os.path.exists(os.path.join(XLMR_DRIVE_DIR, "config.json")):
    print("Found an existing XLM-R checkpoint on Drive — loading instead of retraining.")
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    _model = AutoModelForSequenceClassification.from_pretrained(XLMR_DRIVE_DIR)
    _model.to(_device)  # from_pretrained() does NOT move to GPU automatically like Trainer does during fresh training -
    # without this, a reloaded checkpoint runs prediction on CPU, which is what made evaluation feel stuck/extremely slow.
    xlmr_trained = TrainedModel(
        model=_model,
        tokenizer=AutoTokenizer.from_pretrained(XLMR_DRIVE_DIR),
        output_dir=Path(XLMR_DRIVE_DIR),
    )
    print(f"Loaded onto device: {_device}")
else:
    xlmr_trained = train_transformer(
        XLMR_CHECKPOINT, train_df, val_df,
        output_dir=Path("/content/xlmr_model"),
        text_column=TEXT_COL, label_column=LABEL_COL,
        batch_size=32,  # T4 has headroom for xlm-roberta-base at seq_len=128; fewer, larger steps = faster wall-clock
    )
    print("XLM-R trained. Copying checkpoint to Drive so a disconnect never loses this again...")
    import shutil
    shutil.copytree("/content/xlmr_model", XLMR_DRIVE_DIR, dirs_exist_ok=True)

print("xlmr_trained ready:", xlmr_trained.output_dir)

Found an existing XLM-R checkpoint on Drive — loading instead of retraining.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded onto device: cuda
xlmr_trained ready: /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment_checkpoints/xlmr_model


In [6]:
# Train mBERT (comparison model — this project's own methodology addition,
# not required by FYP2.md §46). Same checkpoint-on-Drive pattern as XLM-R.
bootstrap()
import torch
from src.train_mbert import train_mbert
from src.train_xlmr import TrainedModel, predict as mbert_predict  # same generic predict()
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MBERT_DRIVE_DIR = os.path.join(CKPT_DIR, "mbert_model")

if 'mbert_trained' in globals():
    print("mbert_trained already in memory — skipping.")
elif os.path.exists(os.path.join(MBERT_DRIVE_DIR, "config.json")):
    print("Found an existing mBERT checkpoint on Drive — loading instead of retraining.")
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    _model = AutoModelForSequenceClassification.from_pretrained(MBERT_DRIVE_DIR)
    _model.to(_device)  # same reasoning as the XLM-R cell above - from_pretrained() alone leaves it on CPU
    mbert_trained = TrainedModel(
        model=_model,
        tokenizer=AutoTokenizer.from_pretrained(MBERT_DRIVE_DIR),
        output_dir=Path(MBERT_DRIVE_DIR),
    )
    print(f"Loaded onto device: {_device}")
else:
    mbert_trained = train_mbert(
        train_df, val_df,
        output_dir=Path("/content/mbert_model"),
        text_column=TEXT_COL, label_column=LABEL_COL,
        batch_size=32,  # same reasoning as XLM-R's cell — T4 has headroom at seq_len=128
    )
    print("mBERT trained. Copying checkpoint to Drive so a disconnect never loses this again...")
    import shutil
    shutil.copytree("/content/mbert_model", MBERT_DRIVE_DIR, dirs_exist_ok=True)

print("mbert_trained ready:", mbert_trained.output_dir)

Found an existing mBERT checkpoint on Drive — loading instead of retraining.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded onto device: cuda
mbert_trained ready: /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment_checkpoints/mbert_model


In [7]:
# Evaluate all three models across both evaluation paths (original-
# language vs. translate-to-English first) on the held-out test set.
# Needs tfidf_model/xlmr_trained/mbert_trained from the cells above — if
# any is missing, re-run this whole notebook top to bottom (Runtime > Run
# all); each cell above will reload from its Drive checkpoint rather than
# retraining, so a re-run after a disconnect is fast, not another full
# training run.
bootstrap()
import pandas as pd  # also used by the gold-test-set cell below
from datetime import datetime, timezone
from src import evaluate as evaluate_module

# Python stdlib, not pandas - pd.Timestamp.utcnow() is deprecated in
# recent pandas and risks being removed outright; no reason to depend on
# a pandas API for something this simple.
RUN_VERSION = datetime.now(timezone.utc).strftime("v%Y%m%d-%H%M%S")  # this training run's own model_version

predict_fns = {
    "tfidf_lr": lambda texts: tfidf_predict(tfidf_model, texts),
    "xlmr": lambda texts: xlmr_predict(xlmr_trained, texts),
    "mbert": lambda texts: mbert_predict(mbert_trained, texts),
}

rows = evaluate_module.run_full_comparison(
    predict_fns, test_df,
    model_version=RUN_VERSION,
    dataset_version=DATASET_VERSION,
    training_samples=len(train_df),
    validation_samples=len(val_df),
    text_column=TEXT_COL, label_column=LABEL_COL,
    results_dir=Path("/content/results"),
)

for row in rows:
    print(f"{row.model_type:10s} {row.eval_path:22s} acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")

[evaluate] preparing texts for eval_path=original_language (10769 rows)...
[evaluate] done preparing texts for eval_path=original_language
[evaluate] preparing texts for eval_path=translate_to_english (10769 rows)...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[translate] Helsinki-NLP/opus-mt-zh-en: batch 1/75 (0/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 2/75 (32/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 3/75 (64/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 4/75 (96/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 5/75 (128/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 6/75 (160/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 7/75 (192/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 8/75 (224/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 9/75 (256/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 10/75 (288/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 11/75 (320/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 12/75 (352/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 13/75 (384/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 14/75 (416/2400 done)
[translate] Helsinki-NLP/opus-mt-zh-en: batch 15/75 (448/2400 

In [8]:
!cd /content/insightaid-ai && git log -1 --oneline


8f2a49b (grafted, HEAD -> main, origin/main, origin/HEAD) Fix validateDataset() ignoring synonym column matching (real, pre-existing bug)


## Optional: evaluate against the hand-labelled gold test set

Only run this once `gold_agreed.csv` exists (built from
`gold_test_set/ANNOTATION_GUIDELINES.md`'s double-annotation + agreement
process — a separate, parallel task, not part of this notebook). Per the
guidelines, this is reported **separately** from the public-dataset test
split above, never blended into it — it's the only part of the evaluation
that covers real Malaysian code-mixed text.

In [9]:
bootstrap()
import pandas as pd
GOLD_PATH = os.path.join(DATA_DIR, "..", "gold_test_set", "gold_agreed.csv")
if os.path.exists(GOLD_PATH):
    gold_df = pd.read_csv(GOLD_PATH)
    gold_df = gold_df[gold_df["final_label"].isin(["positive", "neutral", "negative"])]  # drop 'exclude' rows
    print("Gold test set:", len(gold_df), "items")

    gold_rows = []
    for model_type, predict_fn in predict_fns.items():
        y_pred = predict_fn(list(gold_df["text"]))
        computed, row = evaluate_module.evaluate_predictions(
            list(gold_df["final_label"]), y_pred, list(gold_df["language"]),
            model_name=f"insightaid-sentiment-{model_type}",
            model_version=f"{RUN_VERSION}-gold",  # distinct from the public test-set run (unique constraint on model_versions)
            model_type=model_type,
            eval_path="original_language",  # gold set has no translate-first variant
            dataset_version=DATASET_VERSION,
            training_samples=len(train_df),
            validation_samples=len(val_df),
            test_samples=len(gold_df),
            notes="Evaluated against the hand-labelled gold_test_set, not the public dataset's own test split.",
        )
        print(f"{model_type:10s} gold_test_set  acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")
        gold_rows.append(row)
    rows = rows + gold_rows
else:
    print("No gold_agreed.csv yet at", GOLD_PATH, "— skipping. Build it per gold_test_set/ANNOTATION_GUIDELINES.md first.")

No gold_agreed.csv yet at /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment/../gold_test_set/gold_agreed.csv — skipping. Build it per gold_test_set/ANNOTATION_GUIDELINES.md first.


## Record these results in Supabase

Nothing in this app writes to `model_versions` automatically (by design —
only the Supabase **service role** can insert; see
`supabase/migrations/0004_model_versions.sql`). Copy the SQL printed below
into Supabase Dashboard → SQL Editor → New query → Run, once per row.

In [10]:
def sql_literal(value):
    if value is None:
        return "NULL"
    if isinstance(value, (dict, list)):
        return "'" + __import__("json").dumps(value).replace("'", "''") + "'::jsonb"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

from dataclasses import asdict

for row in rows:
    d = asdict(row)
    cols = ", ".join(d.keys())
    vals = ", ".join(sql_literal(v) for v in d.values())
    print(f"INSERT INTO model_versions ({cols}) VALUES ({vals});")
    print()

INSERT INTO model_versions (model_name, model_version, model_type, eval_path, trained_at, dataset_version, training_samples, validation_samples, test_samples, accuracy, precision, recall, f1, per_language_metrics, confusion_matrix, notes) VALUES ('insightaid-sentiment-tfidf_lr', 'v20261020-171546', 'tfidf_lr', 'original_language', '2026-10-20T17:39:58.676619+00:00', 'v20261012-ba90cb55d8f7', 85639, 10763, 10769, 0.7434, 0.7252, 0.7437, 0.7273, '{"en": {"accuracy": 0.7196, "precision": 0.7457, "recall": 0.7181, "f1": 0.7216, "n": 5549}, "ms": {"accuracy": 0.8298, "precision": 0.7198, "recall": 0.641, "f1": 0.663, "n": 2820}, "zh": {"accuracy": 0.6971, "precision": 0.5156, "recall": 0.4647, "f1": 0.4499, "n": 2400}}'::jsonb, '[[3377, 703, 264], [301, 1412, 189], [918, 388, 3217]]'::jsonb, NULL);

INSERT INTO model_versions (model_name, model_version, model_type, eval_path, trained_at, dataset_version, training_samples, validation_samples, test_samples, accuracy, precision, recall, f1, pe

## Save everything back to Drive

Model checkpoints are already saved to Drive incrementally by the cells
above (`CKPT_DIR`) — this just saves the final metrics JSON alongside them.

In [11]:
import shutil

RESULTS_DEST = os.path.join(DATA_DIR, "..", "training_runs", RUN_VERSION)
os.makedirs(RESULTS_DEST, exist_ok=True)
if os.path.exists("/content/results"):
    shutil.copytree("/content/results", os.path.join(RESULTS_DEST, "metrics"), dirs_exist_ok=True)
print("Metrics JSON copied to", RESULTS_DEST)
print("Model checkpoints are at", CKPT_DIR, "(already on Drive, saved as each model finished training).")

Metrics JSON copied to /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment/../training_runs/v20261020-171546
Model checkpoints are at /content/drive/MyDrive/INSIGHTAID_AI/ml_data/sentiment_checkpoints (already on Drive, saved as each model finished training).
